# 01. 데이터 수집 (D1 · D2 · D3 · D5)

| ID | 데이터 | 방법 |
|---|---|---|
| D1 | 태그별 대표 종목 + KRX 전 종목 목록 | `FinanceDataReader` KRX-DESC, 직접 만든 `tag_map.csv` |
| D2 | 대표 종목 뉴스 (약한 라벨) | 네이버 검색 API(NAVER API HUB) — 종목명 검색 |
| D3 | 종목명 없는 경제 뉴스 후보 | 네이버 검색 API — 거시·산업 키워드 검색 |
| D5 | 태그 지수 | 대표 종목 일별 종가(FinanceDataReader) → 동일가중 지수 |

**수집 경로를 명세에서 바꾼 이유**
- `finance.naver.com`, `news.naver.com`, `m.stock.naver.com`의 robots.txt가 모두 `Disallow: /`이고,
  명세가 가정한 종목 뉴스 페이지(`item/news_news.naver`)는 410(Gone)을 반환한다 (2026-10-20 확인).
- 그래서 공식 **네이버 검색 Open API**를 쓴다. 2026년에 NAVER API HUB(네이버 클라우드)로 이관되어
  엔드포인트가 `naverapihub.apigw.ntruss.com/search/v1/news`, 헤더가 `X-NCP-APIGW-API-KEY(-ID)`로 바뀌었다.
- API는 **제목 + 요약(본문 앞 1~2문장)**, 원문 링크, 발행시각을 준다. 본문 전체는 받지 않으므로
  명세의 입력 "제목+본문 앞 3~5문장"은 "제목+요약"으로 대체된다. 원문은 재배포하지 않고 분석용으로만 쓴다.
- KRX 업종 지수(pykrx)는 2026년 현재 KRX 로그인이 필요해, 대표 종목 동일가중 지수로 대체했다(D5).

수집은 오래 걸리므로 터미널에서 `python scripts/collect.py d2|d3`로 돌렸다. 아래 `RUN_COLLECT=True`면 노트북에서 직접 받는다.

In [1]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from src.config import RAW, TAG_STOCKS, EXCLUDED_CONGLOMERATES
from src import crawler

RUN_COLLECT = False
pd.set_option("display.max_colwidth", 80)

## D1. 태그 체계와 대표 종목
태그 15개 × 대표 종목 8~14개. 명세 표를 출발점으로, KRX 상장명(2026-10 기준)과 업종(Industry)을 보고 확정했다.
- 사명 변경 반영: LIG넥스원→LIG디펜스앤에어로스페이스, 엔씨소프트→NC, 티웨이항공→트리니티항공,
  HDC현대산업개발→IPARK현대산업개발, GC녹십자→녹십자 (옛 이름은 약칭으로 남겨 제목 필터·마스킹에 사용)
- HD현대미포는 HD현대중공업에 합병되어 제외
- 다산업 대기업(삼성전자, LG전자, SK, LG화학 등)은 라벨 노이즈가 커서 제외
- 두 산업에 걸친 종목은 태그 2개: 풍산(철강·비철+방산), POSCO홀딩스·SK이노베이션(+2차전지)

In [2]:
listing = crawler.load_krx_listing(refresh=RUN_COLLECT)
tag_map = crawler.build_tag_map(listing)
tag_map.to_csv(RAW / "tag_map.csv", index=False, encoding="utf-8-sig")
print("KRX 상장사:", len(listing), "| 대표 종목(행):", len(tag_map), "| 고유 종목:", tag_map.code.nunique())
print("제외한 대기업:", EXCLUDED_CONGLOMERATES)
tag_map.groupby("tag").agg(n=("name", "size"), stocks=("name", ", ".join))

KRX 상장사: 2867 | 대표 종목(행): 164 | 고유 종목: 161
제외한 대기업: ['삼성전자', 'LG전자', 'LG', 'SK', '삼성물산', '한화', 'LG화학', 'SK스퀘어', 'CJ', 'GS', '롯데지주', '두산', 'LS', '효성']


,n,stocks
tag,,
2차전지,14,"LG에너지솔루션, 에코프로비엠, 에코프로, 엘앤에프, 포스코퓨처엠, 삼성SDI, SK아이이테크놀로지, 천보, 나노신소재, 엔켐, 코스모신..."
건설·부동산,11,"현대건설, GS건설, DL이앤씨, 대우건설, IPARK현대산업개발, 삼성E&A, 계룡건설, 태영건설, 금호건설, 동부건설, 한신공영"
금융(은행·증권·보험),14,"KB금융, 신한지주, 하나금융지주, 우리금융지주, 기업은행, 카카오뱅크, 미래에셋증권, 삼성증권, 키움증권, NH투자증권, 삼성생명, 삼..."
바이오·제약,12,"삼성바이오로직스, 셀트리온, 유한양행, 한미약품, 알테오젠, SK바이오팜, 대웅제약, 종근당, HLB, 녹십자, 리가켐바이오, 에스티팜"
반도체,12,"SK하이닉스, 한미반도체, 리노공업, DB하이텍, 원익IPS, 이오테크닉스, 주성엔지니어링, 하나마이크론, 테크윙, ISC, 피에스케이홀..."
방산,9,"한화에어로스페이스, LIG디펜스앤에어로스페이스, 현대로템, 한국항공우주, 한화시스템, 풍산, 퍼스텍, 빅텍, 스페코"
엔터·미디어,9,"하이브, JYP Ent., CJ ENM, 에스엠, 와이지엔터테인먼트, 스튜디오드래곤, 콘텐트리중앙, 디어유, SBS"
원전·전력기기,11,"두산에너빌리티, HD현대일렉트릭, LS ELECTRIC, 효성중공업, 한전기술, 한전KPS, 비에이치아이, 우리기술, 일진전기, 대한전선..."
인터넷·게임,11,"NAVER, 카카오, 크래프톤, NC, 넷마블, 펄어비스, 카카오게임즈, 위메이드, 시프트업, 컴투스, NHN"


대표 종목의 KRX 표준산업분류(Industry)를 보면 태그와 대체로 일치한다. 일치하지 않는 종목(예: 지주회사로
분류된 고려아연·POSCO홀딩스)은 실제 사업 내용을 보고 유지했다.

In [3]:
tag_map.groupby("tag")["Industry"].agg(lambda s: s.value_counts().head(3).to_dict())

tag
2차전지                                 {'일차전지 및 이차전지 제조업': 6, '기초 화학물질 제조업': 2, '기타 화학제품 제조업': 2}
건설·부동산                                {'토목 건설업': 5, '건물 건설업': 5, '건축기술, 엔지니어링 및 관련 기술 서비스업': 1}
금융(은행·증권·보험)                                           {'기타 금융업': 4, '금융 지원 서비스업': 4, '보험업': 4}
바이오·제약                                   {'의약품 제조업': 6, '자연과학 및 공학 연구개발업': 3, '기초 의약물질 제조업': 2}
반도체                                           {'특수 목적용 기계 제조업': 7, '반도체 제조업': 3, '전자부품 제조업': 1}
방산                              {'항공기,우주선 및 부품 제조업': 2, '무기 및 총포탄 제조업': 2, '그외 기타 운송장비 제조업': 1}
엔터·미디어                   {'오디오물 출판 및 원판 녹음업': 4, '텔레비전 방송업': 2, '영화, 비디오물, 방송프로그램 제작 및 배급업': 2}
원전·전력기기         {'전동기, 발전기 및 전기 변환 · 공급 · 제어 장치 제조업': 5, '일반 목적용 기계 제조업': 2, '절연선 및 케이블 제조업'...
인터넷·게임                             {'소프트웨어 개발 및 공급업': 9, '자료처리, 호스팅, 포털 및 기타 인터넷 정보매개 서비스업': 2}
자동차                                 {'자동차 신품 부품 제조업': 6, '자동차용 엔진 및 자동차 제조업': 2, '고무제품 제조업': 2}
정유·화학                               

## D2 · D3. 네이버 검색 API
- D2: 종목마다 `종목명`(+3글자 이상 약칭 1개)으로 검색, 정렬 `date`/`sim` 각각 최대 1,000건
- 제목에 종목명·약칭이 있는지(`title_has_name`)를 같이 저장 → 02에서 약한 라벨 규칙 적용, E5에서 필터 효과 비교
- D3: 거시 키워드 38개 + 산업 키워드 46개. 산업 키워드만 쓰면 '관련 테마 없음' 기사가 너무 적어진다.

In [4]:
if RUN_COLLECT:
    api = crawler.NaverNewsAPI()
    crawler.collect_stock_news(api, tag_map)
    crawler.collect_general_news(api)

d2_raw = pd.read_csv(RAW / "d2_stock_news_raw.csv", dtype={"code": str})
d3_raw = pd.read_csv(RAW / "d3_general_news_raw.csv")
print(f"D2 원본 {len(d2_raw):,}행 (종목 {d2_raw.code.nunique()}개), 제목 필터 통과 {d2_raw.title_has_name.mean():.1%}")
print(f"D3 원본 {len(d3_raw):,}행 (검색어 {d3_raw['query'].nunique()}개)")
d2_raw.sample(5, random_state=0)[["stock", "tags", "title", "title_has_name", "pub_date", "press"]]

D2 원본 419,512행 (종목 161개), 제목 필터 통과 56.4%
D3 원본 166,598행 (검색어 84개)


,stock,tags,title,title_has_name,pub_date,press
34376,롯데정밀화학,정유·화학,"[이슈] 석화업계 담합 수사, 2명 구속·6명은 기각",False,2026-09-22 16:30,startuptoday.co.kr
50877,DB손해보험,금융(은행·증권·보험),[데이터 뉴스룸] 금융·지주업체 10곳 중 6곳꼴 영업益 상승에 미소…미...,False,2026-09-30 16:54,greened.kr
71648,대한유화,정유·화학,"대한유화 노동조합, 후원금 429만9000원을 초록우산 어린이재단 울산지...",True,2021-12-24 00:12,ksilbo.co.kr
211510,LG생활건강,화장품·소비재,"LG생건, 미국·캐나다 '에이본' 매각…리테일, 디지털 중심으로 북미 사...",True,2026-08-24 18:46,livebiz.today
417563,시프트업,인터넷·게임,[업앤다운]게임주 하락…카카오게임즈↑·위메이드맥스↓,False,2026-10-15 16:52,nspna.com


In [5]:
per_stock = d2_raw.groupby(["stock", "tags"]).agg(rows=("title", "size"), title_hit=("title_has_name", "mean"),
                                                  oldest=("pub_date", "min")).reset_index()
per_stock.sort_values("title_hit").head(10)

,stock,tags,rows,title_hit,oldest
6,HD한국조선해양,조선,4000,0.060500,2026-08-24 07:00
28,SBS,엔터·미디어,2000,0.118000,2026-09-22 08:50
149,한화시스템,방산,2000,0.241000,2026-09-21 07:02
137,하이브,엔터·미디어,4000,0.279750,2023-12-29 20:57
94,에코프로비엠,2차전지,4000,0.281750,2009-05-06 17:55
128,파라다이스,항공·여행,2000,0.289500,2026-06-10 10:40
133,피에스케이홀딩스,반도체,2238,0.290438,2008-05-09 09:18
21,NAVER,인터넷·게임,4000,0.326500,2026-07-28 09:24
24,NH투자증권,금융(은행·증권·보험),4000,0.336500,2026-09-20 10:00
19,LIG디펜스앤에어로스페이스,방산,4000,0.346000,2025-12-29 15:34


제목 포함 비율이 낮은 종목은 종목명이 일반 단어와 겹치거나(검색 결과에 다른 뜻이 섞임),
시황 나열 기사에만 이름이 언급되는 경우다. 이런 기사가 약한 라벨 노이즈의 주된 원천이라 제목 필터로 거른다.

In [6]:
d2_raw["pub_date"] = pd.to_datetime(d2_raw["pub_date"])
d3_raw["pub_date"] = pd.to_datetime(d3_raw["pub_date"])
print("D2 기간:", d2_raw.pub_date.min().date(), "~", d2_raw.pub_date.max().date())
print("D3 기간:", d3_raw.pub_date.min().date(), "~", d3_raw.pub_date.max().date())
pd.concat({"D2": d2_raw.pub_date.dt.to_period("M").value_counts().sort_index(),
           "D3": d3_raw.pub_date.dt.to_period("M").value_counts().sort_index()}, axis=1).fillna(0).astype(int).tail(14)

D2 기간: 2000-03-31 ~ 2026-10-20
D3 기간: 2024-02-21 ~ 2026-10-20


,D2,D3
pub_date,,
2025-09,2697,711
2025-10,2786,623
2025-11,3717,862
2025-12,4351,912
2026-01,4838,1067
2026-02,5795,652
2026-03,8983,1447
2026-04,9209,1414
2026-05,8393,1454


## D5. 태그 지수
대표 종목 일별 종가 → 일간 수익률 동일가중 평균 → 누적 지수. ±30%(가격제한폭)를 넘는 일간 변동은
액면분할 등 수정주가 미반영으로 보고 0으로 처리했다.

In [7]:
if RUN_COLLECT or not (RAW / "d5_prices.csv").exists():
    prices = crawler.collect_prices(tag_map, "2025-10-01")
    prices.to_csv(RAW / "d5_prices.csv", index=False, encoding="utf-8-sig")
prices = pd.read_csv(RAW / "d5_prices.csv", dtype={"code": str}, parse_dates=["date"])
tag_index = crawler.build_tag_index(prices, tag_map)
tag_index.to_csv(RAW / "d5_tag_index.csv", encoding="utf-8-sig")
tag_index.iloc[-1].sort_values(ascending=False).round(2).rename("2025-10 대비 누적")

반도체             2.27
건설·부동산          1.61
원전·전력기기         1.42
금융(은행·증권·보험)    1.34
2차전지            1.29
자동차             1.28
정유·화학           1.28
철강·비철           1.02
바이오·제약          0.94
화장품·소비재         0.93
인터넷·게임          0.85
방산              0.80
항공·여행           0.66
조선              0.56
엔터·미디어          0.52
Name: 2025-10 대비 누적, dtype: float64

## 이 단계에서 알게 된 것
- 네이버 금융·뉴스 웹페이지는 robots.txt로 크롤링이 금지돼 있어 공식 검색 API(NAVER API HUB)로 경로를 바꿨다.
  그 대가로 본문 대신 요약(1~2문장, 평균 160자)만 쓴다. API 호출은 D2 4,198회 + D3 1,666회로 하루 한도(2.5만) 안이다.
- D2 원본 41.9만 행(161종목), D3 원본 16.7만 행(검색어 84개). 종목 검색 결과 중 제목에 종목명이 있는 행은 56%다.
  HD한국조선해양(6%), SBS(12%)처럼 이름이 다른 기사(그룹사 기사, 방송 프로그램)에 자주 섞이는 종목이 낮다.
- 검색 API는 정렬별로 최대 약 1,000건만 주므로 기사 시점이 최근에 몰린다(2026년 9~10월이 대부분).
  `sim` 정렬은 오래된 기사(최대 2000년대)도 섞어 주므로 03에서 최근 12개월로 자른다.
- 태그 지수(D5): 지난 1년 반도체(×2.27)·건설(×1.61)·원전·전력기기(×1.42)가 오르고, 엔터(×0.52)·조선(×0.56)·
  항공·여행(×0.66)이 내렸다. 구성 종목들이 같은 방향으로 움직여 수정주가 문제가 아님을 확인했다.